# 📸 Image Matching Challenge 2022 — EDA & Computer Vision Fundamentals

---

## 🙏 Special Acknowledgement

> **This notebook draws heavily from and builds upon the excellent work in [`image-matching-challenge-2022-eda.ipynb`](https://www.kaggle.com/code/dschettler8845/image-matching-challenge-2022-eda) by Darien Schettler.** 
> Much of the foundational understanding, data exploration, and structural approach in this notebook is inspired by that comprehensive EDA. We extend it with additional computer vision fundamentals from the `computervision-assignment.ipynb` notebook and enhance the visualizations using Plotly dark mode for a more modern, interactive experience.

---

## 📋 Table of Contents

1. **[Introduction & Problem Understanding](#intro)**
2. **[Environment Setup & Imports](#setup)**
3. **[Dataset Overview & Structure](#dataset)**
4. **[Exploratory Data Analysis (EDA)](#eda)**
5. **[Computer Vision Fundamentals](#cv_fundamentals)**
   - 5.1 Feature Detection (SIFT, ORB, AKAZE)
   - 5.2 Feature Matching Basics
   - 5.3 Stereo Vision & Depth Maps
   - 5.4 Fundamental & Essential Matrices
6. **[Structure from Motion (SfM) Pipeline Overview](#sfm)**
7. **[Key Takeaways & Next Steps](#takeaways)**

<a id='intro'></a>
## 1. 🎯 Introduction & Problem Understanding

### What is Image Matching?

Image matching is the process of identifying corresponding points between two or more images of the same scene taken from different viewpoints. This is a fundamental problem in computer vision with applications in:

- **3D Reconstruction** (Structure from Motion)
- **Visual Localization** (GPS-denied navigation)
- **Augmented Reality** (object tracking)
- **Google Maps** (StreetView, 3D models)
- **Cultural Heritage Preservation**

### Competition Task

As detailed in the original EDA notebook by Darien Schettler:

> **Participants are asked to estimate the relative pose of one image with respect to another. For each ID in the test set, you must predict the fundamental matrix between the two views.**

The classical pipeline involves:
1. **Extract Local Features** (keypoints + descriptors)
2. **Match Local Features** between image pairs
3. **Filter Feature Matches** (remove outliers)
4. **Apply RANSAC** (robust estimation of the fundamental matrix)

### Evaluation Metric — mean Average Accuracy (mAA)

The competition uses **mAA** which evaluates:
- **Rotation error** (εR in degrees)
- **Translation error** (εT in meters)

Thresholds range from fine (1° rotation, 0.2m translation) to coarse (10° rotation, 5m translation).

```python
thresholds_r = np.linspace(1, 10, 10)   # In degrees
thresholds_t = np.geomspace(0.2, 5, 10) # In meters
```

<a id='setup'></a>
## 2. ⚙️ Environment Setup & Imports

In [ ]:
# ============================================================
# Core Libraries
# ============================================================
import numpy as np
import pandas as pd
import os
import sys
import glob
import json
import csv
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Computer Vision
# ============================================================
import cv2
from PIL import Image

# ============================================================
# Visualization — Plotly (Dark Mode) + Matplotlib
# ============================================================
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set Plotly dark theme globally
pio.templates.default = 'plotly_dark'

import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

# ============================================================
# Utility
# ============================================================
from collections import Counter, namedtuple
from pathlib import Path

print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")
print(f"OpenCV version:     {cv2.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"\n✅ All imports successful!")

<a id='dataset'></a>
## 3. 📂 Dataset Overview & Structure

### Dataset Files (From the EDA Notebook)

The dataset is organized as follows:

| File/Directory | Description |
|:---|:---|
| `train/*/calibration.csv` | Camera intrinsics (K), rotation (R), translation (T) per image |
| `train/*/pair_covisibility.csv` | Image pairs with covisibility scores and ground truth F matrices |
| `train/scaling_factors.csv` | Scene-level scaling factors to convert poses to meters |
| `train/*/images/` | Training images grouped by scene |
| `test.csv` | Test image pairs (~10,000 pairs) |
| `test_images/` | Test images |
| `sample_submission.csv` | Sample submission format |

### Key Data Fields

- **`camera_intrinsics`**: 3×3 calibration matrix K (focal length, principal point)
- **`rotation_matrix`**: 3×3 rotation matrix R  
- **`translation_vector`**: 3D translation vector T
- **`covisibility`**: Overlap estimate between image pairs (higher = more overlap)
- **`fundamental_matrix`**: 3×3 matrix encoding the epipolar geometry between two views

In [ ]:
# ============================================================
# Dataset Paths — Update these paths based on your environment
# ============================================================
# For Kaggle:
# DATA_DIR = '/kaggle/input/image-matching-challenge-2022/'
# For local:
DATA_DIR = './data/'  # Update this path

# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/image-matching-challenge-2022/'

print(f"📁 Data directory: {DATA_DIR}")
print(f"🖥️  Running on: {'Kaggle' if IS_KAGGLE else 'Local'}")

In [ ]:
# ============================================================
# Load Training Metadata (if available)
# ============================================================
train_csv_path = os.path.join(DATA_DIR, 'train.csv') if IS_KAGGLE else None

if train_csv_path and os.path.exists(train_csv_path):
    train_df = pd.read_csv(train_csv_path)
    print(f"\n📊 Training scenes: {len(train_df)}")
    print(f"\n{train_df.head(20)}")
    
    # Get scene names
    scenes = train_df['scene'].tolist()
    print(f"\n🏛️  Scenes: {scenes}")
else:
    print("⚠️  Training data not found — using demo data for visualization")
    # Demo scene list from the competition
    scenes = [
        'british_museum', 'florence_cathedral_side', 'lincoln_memorial_statue',
        'london_bridge', 'milan_cathedral', 'mount_rushmore', 'notre_dame_front_facade',
        'piazza_san_marco', 'reichstag', 'sacre_coeur', 'sagrada_familia',
        'st_pauls_cathedral', 'st_peters_square', 'taj_mahal', 'temple_nara_japan',
        'trevi_fountain'
    ]
    print(f"\n🏛️  Known competition scenes ({len(scenes)}): {scenes}")

<a id='eda'></a>
## 4. 📊 Exploratory Data Analysis

### 4.1 Scene Distribution Analysis

In [ ]:
# ============================================================
# Visualize Scene Distribution — Plotly Dark Mode
# ============================================================

# Create a visually rich bar chart of scenes
scene_names_display = [s.replace('_', ' ').title() for s in scenes]

# Assign colors from a gradient
colors = px.colors.sequential.Plasma
n_colors = len(colors)
scene_colors = [colors[i % n_colors] for i in range(len(scenes))]

fig = go.Figure(data=[
    go.Bar(
        x=scene_names_display,
        y=list(range(1, len(scenes) + 1)),  # Placeholder counts
        marker=dict(
            color=list(range(len(scenes))),
            colorscale='Plasma',
            showscale=True,
            colorbar=dict(title='Scene Index')
        ),
        text=scene_names_display,
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Index: %{y}<extra></extra>'
    )
])

fig.update_layout(
    title=dict(
        text='🏛️ Competition Scenes Overview',
        font=dict(size=22, color='#E8E8E8'),
        x=0.5
    ),
    xaxis=dict(title='Scene', tickangle=45, tickfont=dict(size=10)),
    yaxis=dict(title='Scene Index'),
    height=500,
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='#1a1a2e'
)

fig.show()

### 4.2 Covisibility Analysis

As noted in the original EDA notebook, **covisibility** measures the overlap between image pairs.
- Higher values → more shared visible content
- The competition recommends using pairs with covisibility ≥ 0.1

In [ ]:
# ============================================================
# Covisibility Distribution — Demo with Synthetic Data
# ============================================================
np.random.seed(42)

# Generate realistic covisibility distributions for demo
demo_scenes = ['British Museum', 'Florence Cathedral', 'Lincoln Memorial', 
               'London Bridge', 'Milan Cathedral', 'Trevi Fountain']

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=demo_scenes,
    horizontal_spacing=0.08,
    vertical_spacing=0.12
)

for i, scene in enumerate(demo_scenes):
    row = i // 3 + 1
    col = i % 3 + 1
    
    # Generate realistic covisibility distribution
    covis = np.random.beta(2, 5, size=500)  # Skewed towards lower values
    mean_covis = np.mean(covis)
    
    fig.add_trace(
        go.Histogram(
            x=covis,
            nbinsx=30,
            name=scene,
            marker_color=px.colors.qualitative.Vivid[i % 11],
            opacity=0.8,
            showlegend=False,
            hovertemplate=f'<b>{scene}</b><br>Covisibility: %{{x:.2f}}<br>Count: %{{y}}<extra></extra>'
        ),
        row=row, col=col
    )
    
    # Add threshold line at 0.1
    fig.add_vline(x=0.1, line_dash='dash', line_color='red', 
                  line_width=2, row=row, col=col)
    
    # Add mean annotation
    fig.add_annotation(
        x=0.7, y=0.85,
        xref=f'x{i+1 if i > 0 else ""}', yref=f'y{i+1 if i > 0 else ""} domain',
        text=f'μ = {mean_covis:.3f}',
        showarrow=False,
        font=dict(size=11, color='cyan'),
        row=row, col=col
    )

fig.update_layout(
    title=dict(
        text='📈 Covisibility Distribution per Scene (Red line = 0.1 threshold)',
        font=dict(size=18, color='#E8E8E8'),
        x=0.5
    ),
    height=600,
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='rgba(0,0,0,0)'
)

fig.show()

### 4.3 Camera Intrinsics Visualization

In [ ]:
# ============================================================
# Camera Intrinsics Explained
# ============================================================

# The Camera Intrinsic Matrix K:
# K = [[fx,  0, cx],
#      [ 0, fy, cy],
#      [ 0,  0,  1]]
#
# fx, fy: focal lengths in pixels
# cx, cy: principal point (optical center)

# Generate demo focal length distributions
np.random.seed(42)
focal_x = np.random.normal(600, 100, 200)
focal_y = np.random.normal(600, 100, 200)
cx = np.random.normal(400, 30, 200)
cy = np.random.normal(300, 30, 200)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Focal Length Distribution (fx, fy)', 'Principal Point Distribution (cx, cy)'],
    horizontal_spacing=0.12
)

fig.add_trace(
    go.Histogram(x=focal_x, name='fx', marker_color='#00d4ff', opacity=0.7, nbinsx=25),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=focal_y, name='fy', marker_color='#ff6b6b', opacity=0.7, nbinsx=25),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=cx, y=cy, mode='markers', name='Principal Points',
               marker=dict(size=6, color='#f9c74f', opacity=0.6),
               hovertemplate='cx: %{x:.1f}<br>cy: %{y:.1f}<extra></extra>'),
    row=1, col=2
)

fig.update_layout(
    title=dict(
        text='📷 Camera Intrinsic Parameters Distribution',
        font=dict(size=18, color='#E8E8E8'),
        x=0.5
    ),
    height=400,
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='rgba(0,0,0,0)'
)

fig.show()

print("\n📝 Camera Intrinsic Matrix K:")
print("   ┌              ┐")
print("   │ fx   0   cx  │")
print("   │  0  fy   cy  │")
print("   │  0   0    1  │")
print("   └              ┘")
print("\n  • fx, fy = focal lengths in pixel units")
print("  • cx, cy = principal point (image center)")

### 4.4 Image Properties Analysis

In [ ]:
# ============================================================
# Image Size & Aspect Ratio Analysis
# ============================================================

# As noted in the EDA notebook: images are resized so longest edge is ~800px
# They may have different aspect ratios (portrait/landscape)

np.random.seed(42)

# Simulate realistic image dimensions
n_images = 300
widths = np.random.choice([800, 600, 533, 640, 720], n_images, p=[0.3, 0.25, 0.15, 0.15, 0.15])
heights = np.random.choice([600, 800, 450, 533, 640], n_images, p=[0.25, 0.3, 0.15, 0.15, 0.15])
aspect_ratios = widths / heights

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Image Width Distribution', 'Image Height Distribution', 'Aspect Ratio Distribution']
)

fig.add_trace(
    go.Histogram(x=widths, nbinsx=20, marker_color='#06d6a0', name='Width'),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=heights, nbinsx=20, marker_color='#118ab2', name='Height'),
    row=1, col=2
)
fig.add_trace(
    go.Histogram(x=aspect_ratios, nbinsx=30, marker_color='#ef476f', name='Aspect Ratio'),
    row=1, col=3
)

fig.update_layout(
    title=dict(
        text='🖼️ Image Dimension Analysis',
        font=dict(size=18, color='#E8E8E8'),
        x=0.5
    ),
    height=400,
    showlegend=False,
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='rgba(0,0,0,0)'
)

fig.show()

<a id='cv_fundamentals'></a>
## 5. 🔬 Computer Vision Fundamentals

This section covers the core CV techniques used in image matching, inspired by techniques shown in `computervision-assignment.ipynb`.

### 5.1 Feature Detection — SIFT, ORB, AKAZE

Feature detection identifies distinctive points in an image that are:
- **Repeatable** — found again under different conditions
- **Distinctive** — uniquely describable
- **Invariant** — robust to transformations (scale, rotation, illumination)

In [ ]:
# ============================================================
# Feature Detection Comparison
# ============================================================

def create_demo_image(seed=42):
    """Create a demo image with distinctive features for demonstration."""
    np.random.seed(seed)
    img = np.zeros((400, 600, 3), dtype=np.uint8)
    
    # Add gradient background
    for y in range(400):
        for x in range(600):
            img[y, x] = [int(50 + x * 0.1), int(30 + y * 0.15), int(80 + (x + y) * 0.05)]
    
    # Add geometric shapes as features
    cv2.rectangle(img, (50, 50), (200, 200), (255, 200, 100), 3)
    cv2.circle(img, (400, 100), 60, (100, 255, 150), 3)
    cv2.rectangle(img, (350, 250), (550, 370), (200, 100, 255), 3)
    cv2.circle(img, (150, 300), 40, (255, 100, 200), 3)
    
    # Add some texture
    for _ in range(50):
        x, y = np.random.randint(0, 600), np.random.randint(0, 400)
        r = np.random.randint(3, 8)
        color = tuple(np.random.randint(100, 255, 3).tolist())
        cv2.circle(img, (x, y), r, color, -1)
    
    return img


def detect_features(img, detector_name='SIFT', max_features=500):
    """Detect features using different detectors."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    if detector_name == 'SIFT':
        detector = cv2.SIFT_create(nfeatures=max_features)
    elif detector_name == 'ORB':
        detector = cv2.ORB_create(nfeatures=max_features)
    elif detector_name == 'AKAZE':
        detector = cv2.AKAZE_create()
    else:
        raise ValueError(f"Unknown detector: {detector_name}")
    
    kp, desc = detector.detectAndCompute(gray, None)
    return kp[:max_features], desc[:max_features] if desc is not None else None


# Create demo image and detect features
demo_img = create_demo_image()

detectors = ['SIFT', 'ORB', 'AKAZE']
results = {}

for det_name in detectors:
    try:
        kp, desc = detect_features(demo_img, det_name)
        results[det_name] = {'keypoints': kp, 'descriptors': desc}
        print(f"✅ {det_name}: {len(kp)} keypoints detected")
        if desc is not None:
            print(f"   Descriptor shape: {desc.shape}")
    except Exception as e:
        print(f"❌ {det_name}: {e}")

In [ ]:
# ============================================================
# Keypoint Response Distribution — Plotly Dark Mode
# ============================================================

fig = make_subplots(
    rows=1, cols=len(results),
    subplot_titles=[f'{name} Keypoint Responses' for name in results.keys()]
)

colors_map = {'SIFT': '#00d4ff', 'ORB': '#ff6b6b', 'AKAZE': '#f9c74f'}

for i, (name, data) in enumerate(results.items(), 1):
    responses = [kp.response for kp in data['keypoints']]
    
    fig.add_trace(
        go.Histogram(
            x=responses, nbinsx=50,
            marker_color=colors_map.get(name, '#00d4ff'),
            opacity=0.8,
            name=name,
            hovertemplate=f'<b>{name}</b><br>Response: %{{x:.4f}}<br>Count: %{{y}}<extra></extra>'
        ),
        row=1, col=i
    )

fig.update_layout(
    title=dict(
        text='🎯 Keypoint Response Distribution by Detector',
        font=dict(size=18, color='#E8E8E8'),
        x=0.5
    ),
    height=400,
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=True
)

fig.show()

In [ ]:
# ============================================================
# Keypoint Size & Angle Analysis
# ============================================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Keypoint Size Distribution', 'Keypoint Spatial Distribution']
)

for name, data in results.items():
    sizes = [kp.size for kp in data['keypoints']]
    xs = [kp.pt[0] for kp in data['keypoints']]
    ys = [kp.pt[1] for kp in data['keypoints']]
    
    fig.add_trace(
        go.Histogram(x=sizes, nbinsx=30, marker_color=colors_map.get(name, '#00d4ff'),
                     opacity=0.6, name=f'{name} Size'),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=xs, y=ys, mode='markers', name=f'{name} Position',
                   marker=dict(size=[s*0.5 for s in sizes], 
                               color=colors_map.get(name, '#00d4ff'),
                               opacity=0.5)),
        row=1, col=2
    )

fig.update_layout(
    title=dict(
        text='📐 Keypoint Properties Analysis',
        font=dict(size=18, color='#E8E8E8'),
        x=0.5
    ),
    height=450,
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='rgba(0,0,0,0)'
)
fig.update_yaxes(autorange='reversed', row=1, col=2)  # Match image coordinates

fig.show()

### 5.2 Feature Matching Basics

Once features are detected in two images, we need to find correspondences:

- **Brute-Force Matching** — Compare every descriptor in image 1 with every descriptor in image 2
- **FLANN-based Matching** — Approximate nearest neighbor for faster matching
- **Ratio Test** (Lowe's) — Keep matches where best match is significantly better than second-best

In [ ]:
# ============================================================
# Feature Matching Demo
# ============================================================

# Create two related images (simulated transformation)
img1 = create_demo_image(42)
# Create a slightly transformed version
rows, cols = img1.shape[:2]
M = cv2.getRotationMatrix2D((cols/2, rows/2), 15, 0.9)  # Rotate 15°, scale 0.9
img2 = cv2.warpAffine(img1, M, (cols, rows))

# Detect SIFT features
sift = cv2.SIFT_create(nfeatures=200)
kp1, desc1 = sift.detectAndCompute(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), None)
kp2, desc2 = sift.detectAndCompute(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), None)

# Brute-Force Matching with Lowe's Ratio Test
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
raw_matches = bf.knnMatch(desc1, desc2, k=2)

# Apply ratio test
good_matches = []
ratio_values = []
for m, n in raw_matches:
    ratio = m.distance / n.distance
    ratio_values.append(ratio)
    if ratio < 0.75:  # Lowe's ratio threshold
        good_matches.append(m)

print(f"Raw matches: {len(raw_matches)}")
print(f"Good matches (ratio < 0.75): {len(good_matches)}")
print(f"Filtering rate: {len(good_matches)/len(raw_matches)*100:.1f}%")

In [ ]:
# ============================================================
# Lowe's Ratio Test Visualization — Plotly Dark Mode
# ============================================================

fig = go.Figure()

# Histogram of ratio values
fig.add_trace(
    go.Histogram(
        x=ratio_values, nbinsx=50,
        marker_color='#00d4ff',
        opacity=0.7,
        name='Distance Ratios',
        hovertemplate='Ratio: %{x:.3f}<br>Count: %{y}<extra></extra>'
    )
)

# Add threshold line
fig.add_vline(x=0.75, line_dash='dash', line_color='#ff6b6b', line_width=3,
              annotation_text='Threshold = 0.75', annotation_position='top right',
              annotation_font_color='#ff6b6b')

fig.update_layout(
    title=dict(
        text="🔍 Lowe's Ratio Test — Match Quality Distribution",
        font=dict(size=18, color='#E8E8E8'),
        x=0.5
    ),
    xaxis_title='Distance Ratio (d1/d2)',
    yaxis_title='Count',
    height=400,
    annotations=[
        dict(x=0.4, y=0.9, xref='paper', yref='paper',
             text=f'✅ Good matches: {len(good_matches)}', showarrow=False,
             font=dict(size=14, color='#06d6a0')),
        dict(x=0.4, y=0.82, xref='paper', yref='paper',
             text=f'❌ Filtered out: {len(raw_matches) - len(good_matches)}', showarrow=False,
             font=dict(size=14, color='#ef476f'))
    ],
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='rgba(0,0,0,0)'
)

fig.show()

### 5.3 Stereo Vision & Depth Maps

From `computervision-assignment.ipynb`, we understand that stereo vision enables depth estimation by:
1. Matching features between left and right images
2. Computing the **disparity map** (pixel offset between corresponding points)
3. Converting disparity to depth using: `depth = (focal_length × baseline) / disparity`

In [ ]:
# ============================================================
# Stereo Vision & Depth Estimation Concepts
# ============================================================

# Demonstrate the relationship between disparity and depth
disparity_values = np.linspace(1, 100, 200)
focal_length = 500  # pixels
baseline = 0.12  # meters (12 cm baseline)

depth_values = (focal_length * baseline) / disparity_values

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=disparity_values, y=depth_values,
        mode='lines',
        line=dict(color='#00d4ff', width=3),
        fill='tozeroy',
        fillcolor='rgba(0, 212, 255, 0.1)',
        name='Depth vs Disparity',
        hovertemplate='Disparity: %{x:.1f} px<br>Depth: %{y:.2f} m<extra></extra>'
    )
)

fig.update_layout(
    title=dict(
        text=f'📏 Depth vs Disparity (f={focal_length}px, B={baseline}m)',
        font=dict(size=18, color='#E8E8E8'),
        x=0.5
    ),
    xaxis_title='Disparity (pixels)',
    yaxis_title='Depth (meters)',
    height=450,
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='rgba(0,0,0,0)',
    annotations=[
        dict(x=0.6, y=0.8, xref='paper', yref='paper',
             text='depth = (f × B) / disparity',
             showarrow=False, font=dict(size=16, color='#f9c74f'),
             bgcolor='rgba(26, 26, 46, 0.8)', bordercolor='#f9c74f', borderwidth=1)
    ]
)

fig.show()

### 5.4 Fundamental & Essential Matrices

The **Fundamental Matrix (F)** encodes the epipolar geometry between two uncalibrated views:
- For corresponding points x and x': `x'ᵀ F x = 0`
- F is a 3×3 matrix with rank 2 (7 degrees of freedom)

The **Essential Matrix (E)** is the calibrated version:
- `E = K₂ᵀ F K₁` where K₁, K₂ are camera intrinsic matrices
- Can be decomposed into rotation R and translation T

In [ ]:
# ============================================================
# Fundamental Matrix Estimation with RANSAC
# ============================================================

# Using our matched keypoints from earlier
if len(good_matches) >= 8:  # Need at least 8 points for F estimation
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    
    # Estimate F using different RANSAC variants
    methods = {
        'RANSAC': cv2.FM_RANSAC,
        'LMEDS': cv2.FM_LMEDS,
    }
    
    # Try USAC_MAGSAC if available (OpenCV 4.5+)
    if hasattr(cv2, 'USAC_MAGSAC'):
        methods['USAC_MAGSAC'] = cv2.USAC_MAGSAC
    
    for method_name, method_flag in methods.items():
        F, mask = cv2.findFundamentalMat(src_pts, dst_pts, method_flag, 1.0, 0.999)
        if F is not None:
            inliers = mask.ravel().sum()
            total = len(mask)
            print(f"\n🔧 {method_name}:")
            print(f"   Inliers: {inliers}/{total} ({inliers/total*100:.1f}%)")
            print(f"   F matrix (flattened): {F[:3,:3].ravel()}")
else:
    print("⚠️  Not enough matches for fundamental matrix estimation")

In [ ]:
# ============================================================
# RANSAC Inlier/Outlier Visualization
# ============================================================

if len(good_matches) >= 8:
    F, mask = cv2.findFundamentalMat(src_pts, dst_pts, cv2.FM_RANSAC, 1.0, 0.999)
    mask_flat = mask.ravel()
    
    inlier_src = src_pts[mask_flat == 1].reshape(-1, 2)
    outlier_src = src_pts[mask_flat == 0].reshape(-1, 2)
    inlier_dst = dst_pts[mask_flat == 1].reshape(-1, 2)
    outlier_dst = dst_pts[mask_flat == 0].reshape(-1, 2)
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Image 1 — Keypoints', 'Image 2 — Keypoints']
    )
    
    # Image 1
    if len(inlier_src) > 0:
        fig.add_trace(
            go.Scatter(x=inlier_src[:, 0], y=inlier_src[:, 1],
                       mode='markers', marker=dict(size=6, color='#06d6a0'),
                       name='Inliers (Img1)'),
            row=1, col=1
        )
    if len(outlier_src) > 0:
        fig.add_trace(
            go.Scatter(x=outlier_src[:, 0], y=outlier_src[:, 1],
                       mode='markers', marker=dict(size=6, color='#ef476f', symbol='x'),
                       name='Outliers (Img1)'),
            row=1, col=1
        )
    
    # Image 2
    if len(inlier_dst) > 0:
        fig.add_trace(
            go.Scatter(x=inlier_dst[:, 0], y=inlier_dst[:, 1],
                       mode='markers', marker=dict(size=6, color='#06d6a0'),
                       name='Inliers (Img2)', showlegend=False),
            row=1, col=2
        )
    if len(outlier_dst) > 0:
        fig.add_trace(
            go.Scatter(x=outlier_dst[:, 0], y=outlier_dst[:, 1],
                       mode='markers', marker=dict(size=6, color='#ef476f', symbol='x'),
                       name='Outliers (Img2)', showlegend=False),
            row=1, col=2
        )
    
    fig.update_yaxes(autorange='reversed')  # Image coordinate system
    
    fig.update_layout(
        title=dict(
            text='⚡ RANSAC Inlier/Outlier Classification',
            font=dict(size=18, color='#E8E8E8'),
            x=0.5
        ),
        height=450,
        paper_bgcolor='#1a1a2e',
        plot_bgcolor='rgba(0,0,0,0)'
    )
    
    fig.show()

<a id='sfm'></a>
## 6. 🏗️ Structure from Motion (SfM) Pipeline Overview

The complete image matching pipeline, as described in the EDA notebook:

In [ ]:
# ============================================================
# SfM Pipeline Visualization
# ============================================================

pipeline_steps = [
    '1. Feature\nExtraction',
    '2. Feature\nMatching',
    '3. Match\nFiltering',
    '4. RANSAC\nF-Matrix',
    '5. Pose\nEstimation',
    '6. 3D\nReconstruction'
]

pipeline_details = [
    'SIFT, ORB, DISK\nSuperPoint',
    'BF, FLANN\nSuperGlue, LoFTR',
    'Ratio Test\nOANet',
    'RANSAC\nMAGSAC++',
    'E = K₂ᵀFK₁\nR, T decomposition',
    'Triangulation\nBundle Adjustment'
]

pipeline_colors = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff', '#ff6b9d', '#c084fc']

fig = go.Figure()

for i, (step, detail, color) in enumerate(zip(pipeline_steps, pipeline_details, pipeline_colors)):
    # Step boxes
    fig.add_shape(
        type='rect',
        x0=i*1.5, y0=0, x1=i*1.5+1.2, y1=1,
        fillcolor=color, opacity=0.8,
        line=dict(color='white', width=2)
    )
    
    # Step label
    fig.add_annotation(
        x=i*1.5+0.6, y=0.65,
        text=step, showarrow=False,
        font=dict(size=11, color='white', family='Arial Black'),
        align='center'
    )
    
    # Detail label
    fig.add_annotation(
        x=i*1.5+0.6, y=0.25,
        text=detail, showarrow=False,
        font=dict(size=9, color='#E8E8E8'),
        align='center'
    )
    
    # Arrow between steps
    if i < len(pipeline_steps) - 1:
        fig.add_annotation(
            x=i*1.5+1.35, y=0.5,
            ax=i*1.5+1.2, ay=0.5,
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=2,
            arrowwidth=2, arrowcolor='#E8E8E8'
        )

fig.update_layout(
    title=dict(
        text='🔄 Image Matching / SfM Pipeline',
        font=dict(size=20, color='#E8E8E8'),
        x=0.5
    ),
    xaxis=dict(visible=False, range=[-0.3, 8.5]),
    yaxis=dict(visible=False, range=[-0.3, 1.3]),
    height=300,
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=False
)

fig.show()

<a id='takeaways'></a>
## 7. 📝 Key Takeaways & Next Steps

### Key Insights (from EDA and CV Fundamentals)

1. **Data Characteristics**:
   - Images are resized to ~800px on the longest edge
   - Variable aspect ratios (portrait & landscape)
   - Covisibility scores help identify meaningful pairs (threshold ≥ 0.1)

2. **Feature Detection**:
   - SIFT provides the most robust features for matching
   - Keypoint response distribution helps filter weak features
   - Modern learned features (SuperPoint, DISK) outperform classical ones

3. **Matching Pipeline**:
   - Lowe's ratio test effectively filters ambiguous matches
   - RANSAC variants (MAGSAC++) provide better inlier detection
   - The fundamental matrix F encodes essential geometric relationships

4. **Evaluation**:
   - mAA metric evaluates both rotation AND translation accuracy
   - Fine thresholds (1°, 0.2m) are challenging and reward precision

### Next Steps → Feature Matching Pipeline Notebook

In the next notebook, we'll implement the full feature matching pipeline with:
- Advanced preprocessing techniques
- Multiple feature detector comparisons
- RANSAC + MAGSAC++ fundamental matrix estimation
- Comprehensive visualization of match quality

In [ ]:
print("\n" + "="*70)
print("🎉 Notebook 1 Complete — EDA & CV Fundamentals")
print("="*70)
print("\n📌 Special thanks to Darien Schettler for the original")
print("   image-matching-challenge-2022-eda.ipynb notebook that")
print("   provided the foundation for this analysis.")
print("\n➡️  Continue to Notebook 2: Feature Matching Pipeline")